In [1]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [2]:
isin = pd.read_excel('Data\\EA_ISINs.xlsx')

In [3]:
unique_isin = tuple(isin['ISIN'])

In [4]:
isin['ISIN'].str[:2].unique()

array(['DE', 'IT', 'FR', 'ES'], dtype=object)

In [5]:
treasury = pd.read_csv('Data\\TreasuryCUSIP.csv')

In [6]:
unique_treasury = tuple(treasury['ISIN'].unique())

In [7]:
hedge_funds = pd.read_csv('key dataframe\\overlap_hedge_funds.csv')

In [8]:
hf_overlap = tuple(hedge_funds['entity_id'].unique())

In [25]:
# Data prep
query = f"""

SELECT 
    s.lender_id AS dealer_id,
    s.lender_name AS dealer_name, 
    COUNT(*) as cnt
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND (s_lender.sector = 'DEALER')
    AND s.security_isin IN {unique_isin}
GROUP BY s.lender_id, s.lender_name
ORDER BY s.lender_id, s.lender_name

"""

df_lending_d = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_25356\464336709.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lending_d = pd.read_sql_query(query, cnxn)


In [26]:
# Data prep
query = f"""

SELECT 
    s.borrower_id AS dealer_id,
    s.borrower_name AS dealer_name, 
    COUNT(*) as cnt
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND (s_borrower.sector = 'DEALER')
    AND s.security_isin IN {unique_isin}
GROUP BY s.borrower_id, s.borrower_name
ORDER BY s.borrower_id, s.borrower_name
"""

df_borrowing_d = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_25356\2310026352.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_borrowing_d = pd.read_sql_query(query, cnxn)


In [27]:
df_d = pd.concat([df_lending_d, df_borrowing_d])[['dealer_id', 'dealer_name']].drop_duplicates().reset_index(drop=True)

In [28]:
df_borrowing_d['dealer_id'].unique()

array(['54930056FHWP7GIWYY08', '5493006QMFDDMYWIAM13',
       '549300FH0WJAPEHTIQ77', '549300ZK53CNGEEI6A29',
       '7LTWFZYICNSX8D621K86', 'DGQCSV2PHVF7I2743539',
       'K6Q0W1PS1L1O4IQL9C32', 'KX1WK48MPD4Y2NCUIZ63',
       'O2RNE8IBXP4R0TD8PU41', 'R0MUWSFPU8MPRO8K5P83',
       'RRAN7P32P0W0YY4XQW79', 'XKZZ2JZF41MRHTR1V493'], dtype=object)

In [29]:
df_d.to_excel('dealer_list.xlsx')

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    LEFT(s.security_isin,2)                                                             AS collateral_country,
    SUM(s.nominal_value)                                                                AS borrowing_volume,
    AVG(repo_rate)                                                                      AS borrowing_rate, 
    AVG(CASE WHEN contractual_maturity < 1 THEN 1 ELSE contractual_maturity END)        AS borrowing_term
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND s_lender.sector = 'DEALER'
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.borrower_id, s.lender_id, collateral_country
ORDER BY s.business_date, s.borrower_id, s.lender_id, collateral_country

"""

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    LEFT(s.security_isin,2)                                                             AS collateral_country,
    SUM(s.nominal_value)                                                                AS lending_volume,
    AVG(repo_rate)                                                                      AS lending_rate, 
    AVG(CASE WHEN contractual_maturity < 1 THEN 1 ELSE contractual_maturity END)        AS lending_term
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.business_date <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND s_borrower.sector = 'DEALER'
    AND s.security_isin IN {unique_isin}
GROUP BY s.business_date, s.lender_id, s.borrower_id, collateral_country
ORDER BY s.business_date, s.lender_id, s.borrower_id, collateral_country
"""

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df = df_borrowing.merge(df_lending, on= ['business_date', 'fund_id', 'dealer_id', 'collateral_country'], how = 'outer')

In [ ]:
df.to_csv('key dataframe\\fund_dealer_country_day.csv')

In [ ]:
df_day = df.groupby(['business_date', 'fund_id', 'dealer_id'], as_index=False)[['borrowing_volume', 'lending_volume']].sum()
df_day.to_csv('key dataframe\\fund_dealer_day.csv')

In [ ]:
# Data prep
query = f"""

SELECT 
    s.valdt AS business_date,
    s.borrower_id AS fund_id,
    s.lender_id AS dealer_id,
    LEFT(s.security_isin,2)                                                             AS collateral_country,
    SUM(s.nominal_value)                                                                AS borrowing_flow
FROM xlab_ecb_prj_sftds_cb_common.hermesf_flow s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.valdt <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_borrower.sector = 'HF'
    AND s_lender.sector = 'DEALER'
    AND s.security_isin IN {unique_isin}
GROUP BY s.valdt, s.borrower_id, s.lender_id, collateral_country
ORDER BY s.valdt, s.borrower_id, s.lender_id, collateral_country
"""

df_borrowing_flow = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f"""

SELECT 
    s.valdt AS business_date,
    s.lender_id AS fund_id,
    s.borrower_id AS dealer_id,
    LEFT(s.security_isin,2)                                                             AS collateral_country,
    SUM(s.nominal_value)                                                                AS lending_flow
FROM xlab_ecb_prj_sftds_cb_common.hermesf_flow s
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_borrower 
    ON s.borrower_id = s_borrower.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_lender 
    ON s.lender_id = s_lender.lei
WHERE s.nominal_ccy IN ('EUR')
    AND s.valdt <= '2025-10-01'
    AND s.gnlcoll = 'SPEC'
    AND s_lender.sector = 'HF'
    AND s_borrower.sector = 'DEALER'
    AND s.security_isin IN {unique_isin}
GROUP BY s.valdt, s.lender_id, s.borrower_id, collateral_country
ORDER BY s.valdt, s.lender_id, s.borrower_id, collateral_country
"""

df_lending_flow = pd.read_sql_query(query, cnxn)

In [ ]:
df_flow = df_borrowing_flow.merge(df_lending_flow, on= ['business_date', 'fund_id', 'dealer_id', 'collateral_country'], how = 'outer')

In [ ]:
df_flow.to_csv('key dataframe\\fund_dealer_country_day_flow.csv')